# Chapter 3: Data Pipeline for Electromagnetic Field Prediction

This chapter demonstrates the implementation of a comprehensive data pipeline for generating high-quality electromagnetic training data using **FEMM (Finite Element Method Magnetics)**.

## Key Features of Our FEMM-Based Pipeline:

### ✅ **Industry-Standard Physics**
- Real finite element analysis using FEMM solver
- No analytical fallbacks - authentic physics only
- Material property modeling (copper, steel, NdFeB magnets)

### ✅ **Multi-Problem Support**
- **Coil Problems**: Circular coil magnetic field analysis
- **Transformer Problems**: E-I core transformer electromagnetic analysis
- **IPM Motor Problems**: Interior permanent magnet motor analysis

### ✅ **FEMM-Only Implementation**

```python
# FEMM is required - no fallbacks
import femm
FEMM_AVAILABLE = True

# System fails fast if FEMM is not available
if not FEMM_AVAILABLE:
    raise RuntimeError("FEMM is required but not available")
```

## FEMM Solver Architecture

Our pipeline includes three specialized solver classes:

### 1. **CoaxialCoilSolver**
- Circular coil geometry creation
- Material property setup (copper, air)
- Boundary condition definition
- Magnetic field extraction at specified points

### 2. **TransformerSolver**  
- E-I core geometry generation
- Primary and secondary winding definition
- Core material modeling (silicon steel)
- Frequency-dependent analysis

### 3. **IPMMotorSolver**
- Stator and rotor geometry creation
- Permanent magnet placement (V-shaped magnets)
- Three-phase circuit definition
- Torque and force calculation

## Data Generation Process

### 1. **Parameter Sampling**
- Latin Hypercube Sampling for efficient parameter space exploration
- Stratified sampling by problem complexity
- Multi-dimensional parameter ranges

### 2. **FEMM Analysis**
- Real electromagnetic field calculations
- Energy and force computation
- Field distribution mapping

### 3. **Result Extraction**
- Magnetic flux density (Bx, By, |B|)
- Forces and torques
- Magnetic energy
- Flux linkage

### 4. **Data Caching and Storage**
- Efficient result caching
- Multi-modal feature extraction
- Performance metrics computation

## Sample FEMM Analysis

```python
# Example: Coaxial Coil Analysis
solver = CoaxialCoilSolver(debug_mode=False)
with solver:
    result = solver.analyze_coil(
        radius=0.05,      # 50mm coil radius
        turns=50,         # Number of turns
        current=5.0,      # 5A current
        wire_radius=0.001 # 1mm wire radius
    )

# Access results
magnetic_field = result.magnetic_field['B_magnitude']
energy = result.energy
analysis_time = result.analysis_time
```

## Key Advantages Over Analytical Methods

### ✅ **Real Physics**
- Accurate field distribution calculations
- Proper material modeling
- Boundary condition effects

### ✅ **Industry Standard**
- Used in actual electromagnetic design
- Validated against experimental data
- Professional engineering tool

### ✅ **Comprehensive Analysis**
- Multi-physics coupling
- Nonlinear material behavior
- Complex geometry support

## Integration with Deep Learning Pipeline

The FEMM-generated data is ready for neural network training:

- **Input Features**: Geometric parameters, material properties, excitation conditions
- **Target Outputs**: Magnetic field distributions, forces, torques, energy
- **Validation**: Cross-validation with known analytical solutions
- **Quality Control**: Automated data validation and filtering

This FEMM-based data pipeline ensures that our deep learning models are trained on **authentic, physics-based electromagnetic data** rather than simplified approximations, leading to more accurate and reliable field prediction capabilities.

In [1]:
# Import FEMM solver and verify availability
from femm_solver import CoaxialCoilSolver, TransformerSolver, IPMMotorSolver, FEMM_AVAILABLE

print(f"FEMM Available: {FEMM_AVAILABLE}")
print("FEMM Solver Classes Imported Successfully")

# This demonstrates the FEMM-only approach - no fallbacks to analytical methods
assert FEMM_AVAILABLE, "FEMM is required for electromagnetic analysis"

FEMM Available: True
FEMM Solver Classes Imported Successfully


In [2]:
# Example parameter ranges for different problem types

# Coil problem parameters
coil_params = {
    'current': {'range': [1.0, 10.0], 'type': 'continuous'},  # Amperes
    'radius': {'range': [0.01, 0.1], 'type': 'continuous'},    # meters
    'turns': {'range': [10, 100], 'type': 'integer'},          # number of turns
    'wire_radius': {'range': [0.0005, 0.002], 'type': 'continuous'}  # meters
}

# Transformer problem parameters
transformer_params = {
    'primary_turns': {'range': [100, 1000], 'type': 'integer'},
    'secondary_turns': {'range': [10, 500], 'type': 'integer'},
    'core_area': {'range': [1e-4, 1e-3], 'type': 'continuous'},  # m²
    'frequency': {'range': [50, 1000], 'type': 'continuous'},    # Hz
    'primary_voltage': {'range': [120, 240], 'type': 'continuous'}  # V
}

# IPM Motor problem parameters
motor_params = {
    'stator_slots': {'range': [12, 48], 'type': 'integer'},
    'rotor_poles': {'range': [4, 16], 'type': 'integer'},
    'stator_outer_radius': {'range': [0.05, 0.15], 'type': 'continuous'},  # m
    'air_gap': {'range': [0.0005, 0.002], 'type': 'continuous'},           # m
    'magnet_strength': {'range': [0.8, 1.4], 'type': 'continuous'},        # Tesla
    'current_amplitude': {'range': [5.0, 20.0], 'type': 'continuous'}      # A
}

print("Parameter ranges defined for all problem types")
print(f"Coil: {len(coil_params)} parameters")
print(f"Transformer: {len(transformer_params)} parameters")
print(f"Motor: {len(motor_params)} parameters")

Parameter ranges defined for all problem types
Coil: 4 parameters
Transformer: 5 parameters
Motor: 6 parameters
